In [1]:
import pandas as pd

df = pd.read_parquet("../data/raw/steam_games.parquet")

# record the row count after each cleaning step (for the Methods section)
log = []
def record(step, data):
    log.append({"step": step, "rows": len(data)})

record("Raw data", df)
print("Duplicate appIDs:", df["appID"].duplicated().sum())

Duplicate appIDs: 0


In [2]:
# 1. remove playtests
df = df[~df["name"].str.contains("playtest", case=False, na=False)]
record("Removed playtests", df)

# 2. remove entries with no genre or no developer
def is_empty(x):
    return x is None or len(x) == 0

df = df[~df["genres"].apply(is_empty) & ~df["developers"].apply(is_empty)]
record("Removed entries missing genre/developer", df)

# 3. keep games with 10+ reviews (Steam's threshold for showing a score)
df["total_reviews"] = df["positive"] + df["negative"]
df = df[df["total_reviews"] >= 10]
record("Kept games with 10+ reviews", df)

pd.DataFrame(log)

,step,rows
0,Raw data,124146
1,Removed playtests,116011
2,Removed entries missing genre/developer,115564
3,Kept games with 10+ reviews,56559


In [3]:
# 4. drop unusable columns
drop_cols = ["tags", "user_score", "peak_ccu", "average_playtime_forever",
             "average_playtime_2weeks", "median_playtime_forever",
             "median_playtime_2weeks", "reviews", "score_rank", "notes"]
df = df.drop(columns=drop_cols)

# 5. free vs paid flag, plus % positive
df["is_free"] = df["price"] == 0
df["pct_positive"] = df["positive"] / df["total_reviews"]

# 6a. convert release_date text to a real date
df["release_date_parsed"] = pd.to_datetime(df["release_date"], format="%b %d, %Y", errors="coerce")
failed = df["release_date_parsed"].isna()
print("Dates that didn't convert:", failed.sum())
print(df.loc[failed, "release_date"].value_counts().head(10))

# 6b. split estimated_owners ("0 - 20000") into low and high numbers
owners = df["estimated_owners"].str.split(" - ", expand=True)
df["owners_low"] = pd.to_numeric(owners[0], errors="coerce")
df["owners_high"] = pd.to_numeric(owners[1], errors="coerce")
print("Owner ranges that didn't convert:", df["owners_low"].isna().sum())

Dates that didn't convert: 0
Series([], Name: count, dtype: int64)
Owner ranges that didn't convert: 0


In [4]:
# use the converted date and drop the leftover text version
df["release_date"] = df["release_date_parsed"]
df = df.drop(columns=["release_date_parsed"])

# drop heavy text/media columns not needed for any question (keeps the file small)
df = df.drop(columns=["detailed_description", "screenshots", "movies", "packages",
                      "support_url", "support_email", "website", "metacritic_url"])

# save the cleaned data and the step log
df.to_parquet("../data/processed/steam_clean.parquet", index=False)
pd.DataFrame(log).to_csv("../data/processed/cleaning_log.csv", index=False)

import os
print(df.shape)
print(round(os.path.getsize("../data/processed/steam_clean.parquet") / 1e6, 1), "MB")

(56559, 28)
13.5 MB
